## NISAR Data Access

This notebook demonstrates how to access and explore NISAR data in Python. NISAR products are distributed as HDF5 files, which can be opened using several different tools depending on the level of detail and functionality needed.

After searching for and downloading NISAR data, this workflow:
- accesses data using `h5py` 
- accesses data using `xarray.Dataset` and `xarray.DataTree`
- accesses data using `ISCE3` 

These approaches highlight different options for accessing NISAR data, allowing users to compare methods based on their research questions and workflow.

## Notebook Overview

1. [Prerequisites](access-prereqs)
1. [Comparing Methods for Accessing NISAR Data](access-comparing)
1. [Selecting a NISAR Procdct](access-selectprod)
2. [Download a NISAR HDF5 File](access-download)
2. [Open the file in H5Py](access-h5py)
3. [Opening the NISAR file in xarray and xarray.DataTree](access-xarray)
4. [Opening the NISAR file in ISCE3](access-isce3)
9. [Summary](access-summary)
10. [Resources and References](access-resources)

<hr>

(access-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/create_software_environment.ipynb) | Necessary | |

- **Rough Notebook Time Estimate**: 10 minutes

<hr>

(access-comparing)=
## 2. Comparing Methods for Accessing NISAR Data

The approaches shown in this notebook provide different ways to access and work with NISAR HDF5 files, each with its own strengths.

| Method | Best For | Pros | Cons |
|--------|----------|------|------|
| **h5py** | Direct, low-level file access | Minimal dependencies, full control over HDF5 structure, can inspect datasets without loading values | Requires long paths, manual navigation, slicing loads data immediately into memory|
| **xarray.Dataset** | Working with one HDF5 group | Labeled arrays, cleaner syntax, easier data handling, supports lazy loading with chunked arrays for large datasets | Opens one group at a time, requires knowing the group path|
| **xarray.Datatree** | Working with multiple HDF5 groups | Preserves full HDF5 structure, supports lazy loading with chunking, useful for navigating unknown files and working across multiple groups | More complex, less efficient for single group analysis|
| **ISCE3** | SAR processing and NISAR-specific workflows | Designed for NISAR data, understands radar geometry and metadata | Does not easily pull all types of metadata, requires additional installation, may not work in all environments |

---

(access-selectprod)=
## 3. Selecting a NISAR Product

NISAR products are identified by a four-letter abbreviation. In the download example below, the product type is controlled by the `processingLevel` field. To learn more about invidicual products, see the [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/).

For this walkthrough, we use:

- **GCOV** — Geocoded Polarimetric Covariance Matrix

Other NISAR products include:

### L0 Products
- **RRSD** — Radar Raw Signal Data

### L1 Products
- **RSLC** — Range-Doppler Single Look Complex
- **RIFG** — Range-Doppler Interferogram
- **ROFF** — Range-Doppler Pixel Offsets
- **RUNW** — Range-Doppler Unwrapped Interferogram

### L2 Products
- **GSLC** — Geocoded Single Look Complex
- **GOFF** — Geocoded Pixel Offsets
- **GCOV** — Geocoded Polarimetric Covariance Matrix
- **GUNW** — Geocoded Unwrapped Interferogram

### L3 Products
- **SME2** — Soil Moisture

#### To search for a different NISAR product, replace `"GCOV"` in the `processingLevel` field with the four-letter abbreviation for your product of interest.

<hr>

(access-download)=

## 4. Download a NISAR HDF5 file

### 4a. Search for NISAR data. For this examlpe, we will use a GCOV data.

In [1]:
import os
import asf_search as asf
from datetime import datetime
from getpass import getpass
from pathlib import Path

# authenticate with Earthdata Login
session = asf.ASFSession()
session.auth_with_creds(input('EDL Username'), getpass('EDL Password'))

# insert start and end dates. Note that dates and times are inclusive. 
start_date = datetime(2025, 12, 4)
end_date = datetime(2025, 12, 5)
area_of_interest = "POLYGON((40.9131 12.3904,41.8891 12.3904,41.8891 13.2454,40.9131 13.2454,40.9131 12.3904))" 

opts = asf.ASFSearchOptions(
    **{
        "maxResults": 250,                    # limit the number of returned results
        "intersectsWith": area_of_interest,   # spatial filter (WKT geometry)
        "start": start_date,                  # temporal filter (start)
        "end": end_date,                      # temporal filter (end)
        
        # to switch NISAR products, change the four letter abbreviation next to processingLevel
        "processingLevel": ["GCOV"],          # product type / processing level
        "dataset": ["NISAR"],                 # mission/dataset name
        "productionConfiguration": ["PR"],    # production config (e.g., provisional)
        "session": session,                   # use authenticated session from above
    }
)

response = asf.search(opts=opts)

pattern = r'^(?!.*QA_STATS).*'

h5_files = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
h5_files

EDL Username jwhite124
EDL Password ········


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001.h5']

### 4b. Download the data

In [2]:
data_dir = Path.home() / "NISAR_data_access_example"
data_dir.mkdir(exist_ok=True)
print(f'data_dir: {data_dir}')

#download files
asf.download_urls(h5_files, data_dir, session=session, processes=4)

data_dir: /home/jovyan/NISAR_data_access_example


<hr>

(access-h5py)=
## 5. Opening NISAR Data with h5py

The `h5py` library allows you to access HDF5 files directly in Python. NISAR products are stored as hierarchical datasets, similar to folders and files.

In this example, we open a NISAR file, navigate to the metadata group, read the `incidenceAngle` dataset and print the shape.

First, we inspect the dataset shape without loading the data values into memory. Then, we load the full dataset using `[:]`. Finally, we read a smaller subset and calculate its mean.

Note: In `h5py`, accessing a dataset object does not load the data values into memory. Data are loaded when values are explicitly accessed, such as with `[:]` or slicing. Once sliced, the result is loaded into memory as a NumPy array.

### 5a. Inspect dataset without loading into memory

In [3]:
%%time

import h5py

h5_h5py = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

with h5py.File(h5_h5py, "r") as f:
    incidence = f["/science/LSAR/GCOV/metadata/radarGrid/incidenceAngle"]
    print("incidenceAngle shape:", incidence.shape)

incidenceAngle shape: (21, 680, 695)
CPU times: user 11.4 ms, sys: 10.5 ms, total: 21.9 ms
Wall time: 21.1 ms


### 5b. Full load: read dataset into memory

In [4]:
%%time

import h5py

h5_h5py = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

with h5py.File(h5_h5py, "r") as f:
    incidence = f["/science/LSAR/GCOV/metadata/radarGrid/incidenceAngle"][:]

print("incidenceAngle shape:", incidence.shape)

incidenceAngle shape: (21, 680, 695)
CPU times: user 130 ms, sys: 40.1 ms, total: 170 ms
Wall time: 169 ms


### 5c. Subset and compute mean
Note: The subset mean is used here only to demonstrate how data are loaded and processed. Because `incidenceAngle` is a 3D metadata grid, scientific interpretation should be done carefully, usually for a specific height level and spatial region.

In [5]:
%%time
# Subset load: read one height level and a small spatial subset, then compute a statistic

import h5py
import numpy as np

h5_h5py = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

with h5py.File(h5_h5py, "r") as f:
    incidence = f["/science/LSAR/GCOV/metadata/radarGrid/incidenceAngle"]

    subset = incidence[0, 0:100, 0:100]
    subset_mean = np.nanmean(subset)

print("subset mean:", subset_mean)

subset mean: 46.621464
CPU times: user 3.64 ms, sys: 2.96 ms, total: 6.6 ms
Wall time: 5.94 ms


<hr>

(access-xarray)=

## 6. Opening NISAR Data with xarray.Dataset and xarray.DataTree

The `xarray` package provides tools for working with labeled arrays. Here, we show two xarray data structures:

- `xarray.Dataset`: useful when opening one specific group in a NISAR HDF5 file.
- `xarray.DataTree`: useful when opening the full hierarchical file structure.

Both can support lazy loading workflows when opened with `chunks={}`.

### 6a. Opening NISAR Data with xarray.Dataset

Here, we open the NISAR metadata `radarGrid` group as an `xarray.Dataset`. This lets us access variables in the group, such as `incidenceAngle`, using shorter variable names.

We open the dataset with `chunks={}`, which creates a Dask-backed xarray object. This allows operations to be queued without immediately loading the full array into memory.

Note that `open_dataset` only works for one group at a time. When accessing more than one group, use `open_datatree`. 

In [6]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

import xarray as xr

h5_xarray = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

ds = xr.open_dataset(
    h5_xarray,
    group="/science/LSAR/GCOV/metadata/radarGrid",
    chunks={}
)

incidence = ds["incidenceAngle"]

print("incidenceAngle shape:", incidence.shape)
incidence

incidenceAngle shape: (21, 680, 695)


<xarray.DataArray 'incidenceAngle' (heightAboveEllipsoid: 21,
                                    yCoordinates: 680, xCoordinates: 695)> Size: 40MB
dask.array<open_dataset-incidenceAngle, shape=(21, 680, 695), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
Coordinates:
  * heightAboveEllipsoid  (heightAboveEllipsoid) float64 168B -1e+03 ... 9e+03
  * yCoordinates          (yCoordinates) float64 5kB 1.591e+06 ... 1.251e+06
  * xCoordinates          (xCoordinates) float64 6kB 6.018e+05 ... 9.488e+05
Attributes:
    description:   Incidence angle is defined as the angle between the LOS ve...
    grid_mapping:  projection
    long_name:     Incidence angle
    units:         degrees
    valid_max:     90.0
    valid_min:     0.0

**Here, we select one height level and a small spatial subset, then calculate the mean. This operation is queued, but not computed yet.**

In [7]:
%%time

subset_mean = (
    ds["incidenceAngle"]
    .isel(heightAboveEllipsoid=0, yCoordinates=slice(0, 100), xCoordinates=slice(0, 100))
    .mean()
)

subset_mean

CPU times: user 3.42 ms, sys: 840 μs, total: 4.26 ms
Wall time: 3.79 ms


<xarray.DataArray 'incidenceAngle' ()> Size: 4B
dask.array<mean_agg-aggregate, shape=(), dtype=float32, chunksize=(), chunktype=numpy.ndarray>
Coordinates:
    heightAboveEllipsoid  float64 8B -1e+03
Attributes:
    description:   Incidence angle is defined as the angle between the LOS ve...
    grid_mapping:  projection
    long_name:     Incidence angle
    units:         degrees
    valid_max:     90.0
    valid_min:     0.0

**Calling `.compute()` triggers the actual read and calculation.**

In [8]:
%%time

subset_mean_value = subset_mean.compute()

print("subset mean:", subset_mean_value.values)

subset mean: 46.621464
CPU times: user 93.7 ms, sys: 28 ms, total: 122 ms
Wall time: 121 ms


In [9]:
ds.close()

## 6b. Opening NISAR Data with xarray DataTree

An `xarray.DataTree` preserves the full hierarchical structure of the NISAR HDF5 file. This is useful when you want to work across multiple groups.

Here, we open the full file as a tree-like object and access the same `incidenceAngle` dataset through its group path.

In [10]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

import xarray as xr

h5_datatree = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

dt = xr.open_datatree(
    h5_datatree,
    engine="h5netcdf",
    chunks={}
)

incidence = dt["/science/LSAR/GCOV/metadata/radarGrid"]["incidenceAngle"]

print("incidenceAngle shape:", incidence.shape)
dt

incidenceAngle shape: (21, 680, 695)


<xarray.DataTree>
Group: /
│   Attributes:
│       Conventions:         CF-1.7
│       contact:             nisar-sds-ops@jpl.nasa.gov
│       institution:         NASA JPL
│       mission_name:        NISAR
│       reference_document:  D-102274 NISAR NASA SDS Product Specification Level-...
│       title:               NISAR L2 GCOV Product
└── Group: /science
    └── Group: /science/LSAR
        ├── Group: /science/LSAR/GCOV
        │   ├── Group: /science/LSAR/GCOV/grids
        │   │   ├── Group: /science/LSAR/GCOV/grids/frequencyA
        │   │   │       Dimensions:                (yCoordinates: 16704, xCoordinates: 17064,
        │   │   │                                   phony_dim_0: 2)
        │   │   │       Coordinates:
        │   │   │         * yCoordinates           (yCoordinates) float64 134kB 1.588e+06 ... 1.254e+06
        │   │   │         * xCoordinates           (xCoordinates) float64 137kB 6.041e+05 ... 9.454e+05
        │   │   │       Dimensions without coordinates: phony_dim_0
        │   │   │       Data variables:
        │   │   │           HHHH                   (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │   │           HVHV                   (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │   │           listOfCovarianceTerms  (phony_dim_0) |S4 8B dask.array<chunksize=(2,), meta=np.ndarray>
        │   │   │           listOfPolarizations    (phony_dim_0) |S2 4B dask.array<chunksize=(2,), meta=np.ndarray>
        │   │   │           mask                   (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │   │           numberOfLooks          (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │   │           numberOfSubSwaths      uint8 1B ...
        │   │   │           projection             uint32 4B ...
        │   │   │           rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │   │           xCoordinateSpacing     float64 8B ...
        │   │   │           yCoordinateSpacing     float64 8B ...
        │   │   └── Group: /science/LSAR/GCOV/grids/frequencyB
        │   │           Dimensions:                (yCoordinates: 4176, xCoordinates: 4266,
        │   │                                       phony_dim_1: 2)
        │   │           Coordinates:
        │   │             * yCoordinates           (yCoordinates) float64 33kB 1.588e+06 ... 1.254e+06
        │   │             * xCoordinates           (xCoordinates) float64 34kB 6.041e+05 ... 9.453e+05
        │   │           Dimensions without coordinates: phony_dim_1
        │   │           Data variables:
        │   │               HHHH                   (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │               HVHV                   (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │               listOfCovarianceTerms  (phony_dim_1) |S4 8B dask.array<chunksize=(2,), meta=np.ndarray>
        │   │               listOfPolarizations    (phony_dim_1) |S2 4B dask.array<chunksize=(2,), meta=np.ndarray>
        │   │               mask                   (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │               numberOfLooks          (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │               numberOfSubSwaths      uint8 1B ...
        │   │               projection             uint32 4B ...
        │   │               rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(512, 512), meta=np.ndarray>
        │   │               xCoordinateSpacing     float64 8B ...
        │   │           

In [11]:
dt.close()

<hr>

(access-isce3)=
## 7. ISCE3

The NISAR product reader (part of ISCE3) provides a more complex interface for working with NISAR data. It understands the structure of NISAR products and includes built-in attributes and methods for accessing product information and image datasets.

Unlike `h5py` and `xarray`, which access datasets directly through the HDF5 structure, `ISCE3` interacts with the data through class methods and member variables.

## 7a. Opening NISAR Metadata with ISCE3

In this example, we first access general product information, `polarizations` metadata, using reader member variables.

These values come from the product metadata, but are accessed differently than in the previous examples. Instead of navigating the file structure (as with `h5py` or `xarray`), `ISCE3` exposes selected metadata through object attributes.

While some metadata can be accessed this way, many metadata datasets (such as `incidenceAngle`) are not available through a simple or consistent interface in ISCE3. For this reason, `h5py` or `xarray` may be more ideal for accessing metadata.

In [12]:
from nisar.products.readers import open_product

h5_isce3 = list(data_dir.glob("NISAR_L2_*.h5"))[0]

product = open_product(h5_isce3)

print("polarizations:", product.polarizations)

polarizations: {'A': ['HH', 'HV'], 'B': ['HH', 'HV']}


## 7b. Using ISCE3 to read image data 

Next, we use the `ISCE3` reader to access an image dataset using the `getImageDataset()` method.

This method provides a convenient way to access image layers (such as polarization datasets), which are central to many SAR and NISAR workflows. This type of access is a more common use of ISCE3, as it is designed to work with image data and SAR-specific processing rather than general metadata retrieval.

In [13]:
from nisar.products.readers import open_product

h5_hvhv = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

hvhv_data = open_product(h5_hvhv)

hvhv = hvhv_data.getImageDataset(frequency="A", polarization="HVHV")

print("HVHV shape:", hvhv[:, :].shape)

HVHV shape: (16704, 17064)


<hr>

(access-summary)=
## 8. Summary

You have now three methods and examples for opening NISAR data.

<hr>

(access-resources)=
## 9. Resources and references
- [HDF5 Documentation](https://support.hdfgroup.org/documentation/)
- [h5py](https://github.com/h5py/h5py)
- [xarray](https://docs.xarray.dev/en/stable/)
- [xarray.DataTree](https://docs.xarray.dev/en/latest/generated/xarray.DataTree.html)
- [xarray.Dataset](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.html)
- [isce3](https://github.com/isce-framework/isce3)
- [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/)

**Authors:** Julia White, Alex Lewandowski